### Settings

In [7]:
import os
from pathlib import Path
from datetime import date

from client import create_client, get_logger
from extract import dry_run, extract_single_day, extract_date_range, QUERY_TEMPLATE
from transform import optimize_types

KEY_PATH=os.environ.get("GCP_KEY_PATH", "gcp-key.json")
OUTPUT_DIR = Path("../data/daily_agg")

client = create_client("../"+KEY_PATH)
logger = get_logger("extract")

print("Client ready:", client.project)

Client ready: gen-lang-client-0791667039


### Dry Run

In [4]:
test_date = "20260314"
info = dry_run(client, test_date)
print(f"Bytes processed: {info["bytes_processed"] / 1024**3:.2f}GB")
print(f"Estimated cost: ${info["estimated_cost_usd"]:.4f}")

Bytes processed: 0.10GB
Estimated cost: $0.0005


### 1 day test

In [5]:
df_test = extract_single_day(client, test_date)
df_test = optimize_types(df_test)

print(f"Rows: {len(df_test):,}")
print(f"Columns: {list(df_test.columns)}")
print(f"Dtypes:\n{df_test.dtypes}")
df_test.head()

Rows: 1,286,115
Columns: ['actor_id', 'repo_id', 'type', 'cnt']
Dtypes:
actor_id       Int32
repo_id        Int64
type        category
cnt            Int64
dtype: object


,actor_id,repo_id,type,cnt
0,4925587,1167025028,GollumEvent,1
1,14190352,1182012997,PublicEvent,1
2,35613825,1106814127,CommitCommentEvent,5
3,35613825,1162396491,CommitCommentEvent,2
4,41898282,1174498009,CommitCommentEvent,1


In [9]:
# parquet 크기 확인
test_path=OUTPUT_DIR/"_test.parquet"
test_path.parent.mkdir(parents=True, exist_ok=True)
df_test.to_parquet(test_path, index=False)

size_mb = test_path.stat().st_size / 1024**2
print(f"Parquet size: {size_mb:.1f}MB")

# test file 정리
test_path.unlink()

Parquet size: 12.8MB


In [25]:
START_DATE = date(2026, 2, 15)
END_DATE = date(2026, 3, 14) 

saved_files = extract_date_range(client, START_DATE, END_DATE, OUTPUT_DIR, logger)
print(f"\nTotal files: {len(saved_files)}")

2026-03-31 21:33:15,646 - extract - INFO - Processing 20260215
2026-03-31 21:35:05,435 - extract - INFO - Saved ../data/daily_agg/20260215.parquet (12.75 MB)
2026-03-31 21:35:05,436 - extract - INFO - Processing 20260216
2026-03-31 21:37:01,132 - extract - INFO - Saved ../data/daily_agg/20260216.parquet (14.93 MB)
2026-03-31 21:37:01,133 - extract - INFO - Processing 20260217
2026-03-31 21:38:55,920 - extract - INFO - Saved ../data/daily_agg/20260217.parquet (14.96 MB)
2026-03-31 21:38:55,922 - extract - INFO - Processing 20260218


KeyboardInterrupt: 

### verification

In [26]:
import pandas as pd
parquet_files = sorted(OUTPUT_DIR.glob("*.parquet"))
print(f"Parquet files: {len(parquet_files)}")

df_all=pd.read_parquet(OUTPUT_DIR)
df_all=optimize_types(df_all)

print(f"Total rows:     {len(df_all):,}")
print(f"Unique actors:  {df_all['actor_id'].nunique():,}")
print(f"Unique repos:   {df_all['repo_id'].nunique():,}")
print(f"Event types:    {sorted(df_all['type'].unique())}")
print(f"\nMemory usage:   {df_all.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Parquet files: 3
Total rows:     3,932,835
Unique actors:  1,334,400
Unique repos:   1,868,383
Event types:    ['CommitCommentEvent', 'CreateEvent', 'DeleteEvent', 'DiscussionEvent', 'ForkEvent', 'GollumEvent', 'IssueCommentEvent', 'IssuesEvent', 'MemberEvent', 'PublicEvent', 'PullRequestEvent', 'PullRequestReviewCommentEvent', 'PullRequestReviewEvent', 'PushEvent', 'ReleaseEvent', 'WatchEvent']

Memory usage:   120.0 MB
